Step 7: Preprocessing pipeline and baseline models

Why a pipeline: It learns imputation values and scaling from train only, then applies the same rules to test.
If skipped: Filling or scaling with the full data lets test information leak into training, and your scores look better than reality.

7A. Load and separate columns

In [4]:
import pandas as pd
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, average_precision_score, roc_auc_score, confusion_matrix

train = pd.read_csv("../data/processed/train.csv")
test = pd.read_csv("../data/processed/test.csv")

y_train, y_test = train.pop("is_late"), test.pop("is_late")
X_train = train.drop(columns=["order_purchase_timestamp"])  # raw timestamp can't go in the model; month/hour/weekday already capture it
X_test = test.drop(columns = ["order_purchase_timestamp"])

cat_cols = ["customer_state", "seller_state", "category", "payment_type"]
num_cols = [c for c in X_train.columns if c not in cat_cols]
print(len(num_cols), num_cols)   # expect 16

16 ['estimated_days', 'n_items', 'n_sellers', 'total_price', 'total_freight', 'total_weight_g', 'total_volume_cm3', 'max_installments', 'distance_km', 'same_state', 'purchase_month', 'purchase_weekday', 'purchase_hour', 'freight_ratio', 'is_weekend', 'price_per_item']


7B. Build the preprocessor

In [2]:
preprocess = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")),
                      ("sc",  StandardScaler())]), num_cols),
    ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                      ("oh",  OneHotEncoder(handle_unknown="ignore", min_frequency=100))]), cat_cols),
])

| Part |	Why | If skipped |  
|----------| ---------------------------- | -------------------------- |  
Median imputer	| Fills the nulls we kept (distance, weight, payment). The median resists outliers.	| The model crashes on NaN  |    
| StandardScaler |	Puts price (R$ 1000s) and same_state (0/1) on one scale. Needed for logistic regression. |	Large-valued columns dominate  |  
| OneHotEncoder |	Turns state/category text into 0/1 columns |	The model can’t read text  |  
| handle_unknown="ignore" |	A new category in test won’t crash |	Crash on unseen values |    
| min_frequency=100 | 	Rare categories group into one “other” column |	Hundreds of noisy columns |    

7C. Evaluate two baselines

In [3]:
def evaluate(name, clf):
    model = Pipeline([("prep", preprocess), ("clf", clf)])
    model.fit(X_train, y_train)
    proba = model.predict_proba(X_test)[:, 1]
    pred = (proba >= 0.5).astype(int)
    print(f"\n===== {name} =====")
    print(classification_report(y_test, pred, digits=3))
    print("PR-AUC :", round(average_precision_score(y_test, proba), 3),
          "| ROC-AUC:", round(roc_auc_score(y_test, proba), 3))
    print(confusion_matrix(y_test, pred))
    return model

dummy = evaluate("Dummy: always on-time", DummyClassifier(strategy="most_frequent"))
logit = evaluate("Logistic Regression", LogisticRegression(max_iter=1000, class_weight="balanced"))


===== Dummy: always on-time =====
              precision    recall  f1-score   support

           0      0.947     1.000     0.973     18221
           1      0.000     0.000     0.000      1020

    accuracy                          0.947     19241
   macro avg      0.473     0.500     0.486     19241
weighted avg      0.897     0.947     0.921     19241

PR-AUC : 0.053 | ROC-AUC: 0.5
[[18221     0]
 [ 1020     0]]


d:\projects_on_DataAnalyst\olist-delivery-prediction\venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
d:\projects_on_DataAnalyst\olist-delivery-prediction\venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
d:\projects_on_DataAnalyst\olist-delivery-prediction\venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _war


===== Logistic Regression =====
              precision    recall  f1-score   support

           0      0.981     0.548     0.703     18221
           1      0.091     0.811     0.164      1020

    accuracy                          0.561     19241
   macro avg      0.536     0.679     0.433     19241
weighted avg      0.934     0.561     0.674     19241

PR-AUC : 0.116 | ROC-AUC: 0.722
[[9976 8245]
 [ 193  827]]


Why these two

`Dummy`: Predicts “on-time” for everything. Expect about 94.7% accuracy and 0% recall. This proves accuracy is a trap, and it is the floor every real model must beat.  
`Logistic Regression`: A simple, fast, explainable baseline. If XGBoost can’t beat it clearly, the extra complexity isn’t worth it.  
`class_weight="balanced"`: Without it, the model sees 92% on-time and learns to say “on-time” every time. This makes the rare late class count more in training.

7D. Commit

git add .  
git commit -m "Step 7: preprocessing pipeline and baseline models"  
git push

Step 8: Random Forest and XGBoost

Why these models: Our data is tabular, with mixed types and nonlinear patterns (distance matters more past a certain point, and state interacts with category). Tree models capture this automatically. If they don’t beat Logistic Regression clearly, the extra complexity isn’t worth it.

8A. Better comparison function

In [5]:
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

def evaluate(name, clf):
    model = Pipeline([("prep", preprocess), ("clf", clf)])
    model.fit(X_train, y_train)
    proba = model.predict_proba(X_test)[:, 1]
    yt = y_test.values
    top = proba >= np.quantile(proba, 0.9)          # the riskiest 10% of orders
    return model, {"model": name,
                   "PR-AUC": round(average_precision_score(yt, proba), 3),
                   "ROC-AUC": round(roc_auc_score(yt, proba), 3),
                   "precision@top10%": round(yt[top].mean(), 3),
                   "recall@top10%": round(yt[top].sum() / yt.sum(), 3)}

Why “top 10%”: An operations team can only check or expedite a limited number of orders. This metric answers: “if we review the 10% riskiest orders, how many late ones do we catch?” It is also threshold-free.


8B. Train and compare

In [6]:
spw = (y_train == 0).sum() / (y_train == 1).sum()   # about 10: weight of late class

models = {
    "Logistic": LogisticRegression(max_iter=1000, class_weight="balanced"),
    "RandomForest": RandomForestClassifier(n_estimators=300, max_depth=12, min_samples_leaf=50,
                                           class_weight="balanced_subsample", n_jobs=-1, random_state=42),
    "XGBoost": XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.05, subsample=0.8,
                             colsample_bytree=0.8, scale_pos_weight=spw, n_jobs=-1, random_state=42),
}

fitted, rows = {}, []
for name, clf in models.items():
    fitted[name], r = evaluate(name, clf)
    rows.append(r)

results = pd.DataFrame(rows).set_index("model")
print(results)
print("Random baseline: PR-AUC 0.053, precision@top10% 0.053")

              PR-AUC  ROC-AUC  precision@top10%  recall@top10%
model                                                         
Logistic       0.116    0.722             0.112          0.212
RandomForest   0.062    0.531             0.076          0.143
XGBoost        0.107    0.713             0.116          0.219
Random baseline: PR-AUC 0.053, precision@top10% 0.053


| Setting |	Why	| If skipped |  
|---------|--------| ---------|  
|max_depth, min_samples_leaf|	Limit tree size, which prevents memorizing train data|	Overfit: great train score, poor test score |
|scale_pos_weight	|Makes the rare late class count about 10x more	| The model ignores late orders|
|random_state=42	|Same result every run	|Results change each run, and the report can’t be reproduced|
|Same preprocess for all|	A fair comparison|	You can’t tell which model won|

Why not other algorithms: KNN is slow on 77k rows and needs scaling. SVM is slow at this size and gives no easy probabilities. Neural networks need more data and are hard to explain on this tabular data.

8C. Save the table and commit

In [7]:
results.to_csv("../reports/model_comparison.csv")